# Introdução ao Algoritmo de Árvore de Decisão

Uma Árvore de Decisão é um modelo preditivo que mapeia as observações sobre um item para conclusões sobre o valor alvo do item. Ela funciona de maneira hierárquica, onde as decisões são tomadas em nós internos com base em critérios específicos, até se chegar a uma folha que representa a decisão final.

# Preparação dos Dados

Antes de treinar o modelo, precisamos preparar os dados. Nesta etapa, carregamos o dataset, realizamos o pré-processamento necessário (como a codificação de variáveis categóricas) e dividimos os dados em conjuntos de treino e teste.


## Carregamento e Exploração do Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Importar e visualizar os dados
dados_cogumelos = pd.read_csv('./cogumelos.csv')
dados_cogumelos.head()

### Descrição do dataset
O dataset contem 8124 amostras de variedades de cogumelos. cada amostra está descrita com os seus respetivos atributos e rotulado com uma saida quedetermina se o cogumelo é comestivel ou não.
#### A entrada contém 22 atributos:

* **cap-shape:** bell=b, conical=c, convex=x, flat=f, knobbed=k, sunken=s

* **cap-surface:** fibrous=f, grooves=g, scaly=y, smooth=s

* **cap-color:** brown=n, buff=b, cinnamon=c, gray=g, green=r,  pink=p, purple=u, red=e, white=w, yellow=y

* **bruises?** bruises=t, no=f
  
* **odor:** almond=a, anise=l, creosote=c, fishy=y, foul=f,  musty=m, none=n, pungent=p, spicy=s
  
* **gill-attachment:** attached=a, descending=d, free=f, notched=n
  
* **gill-spacing:** close=c, crowded=w, distant=d
  
* **gill-size:** broad=b, narrow=n
  
* **gill-color:** black=k, brown=n, buff=b, chocolate=h, gray=g,  green=r, orange=o, pink=p, purple=u, red=e, white=w, yellow=y
  
* **stalk-shape:** enlarging=e, tapering=t

* **stalk-root:** bulbous=b, club=c, cup=u, equal=e, rhizomorphs=z, rooted=r, missing=?
  
* **stalk-surface-above-ring:** fibrous=f, scaly=y, silky=k, smooth=s
  
* **stalk-surface-below-ring:** fibrous=f, scaly=y, silky=k, smooth=s
  
* **stalk-color-above-ring:** brown=n, buff=b, cinnamon=c, gray=g, orange=o, pink=p, red=e, white=w, yellow=y
  
* **stalk-color-below-ring:** brown=n, buff=b, cinnamon=c, gray=g, orange=o, pink=p, red=e, white=w, yellow=y
  
* **veil-type:** partial=p, universal=u
  
* **veil-color:** brown=n, orange=o, white=w, yellow=y

* **ring-number:** none=n, one=o, two=t

* **ring-type:** cobwebby=c, evanescent=e, flaring=f, large=l, none=n, pendant=p, sheathing=s, zone=z
  
* **spore-print-color:** black=k, brown=n, buff=b, chocolate=h, green=r,  orange=o, purple=u, white=w, yellow=y
  
* **population:** abundant=a, clustered=c, numerous=n, scattered=s, several=v, solitary=y
  
* **habitat:** grasses=g, leaves=l, meadows=m, paths=p,  urban=u, waste=w, woods=d


### A saída para classificação indica se o cogumelo é comestível ou não:
* **poisonous:** p=poisonous, e=edible


In [ ]:
# Visualização geral das informações do dataset
# Mostra a estrutura do DataFrame, incluindo o tipo de dados de cada coluna e valores nulos.
dados_cogumelos.info()

In [ ]:
# Descrição estatística básica das colunas do dataset.
# Gera estatísticas descritivas que resumem a tendência central, dispersão e forma da distribuição de um conjunto de dados.
dados_cogumelos.describe()

In [ ]:
# Verificar as dimensões do dataframe (número de linhas e colunas).
# Dimensões do dataframe
# O shape retorna as dimensões do DataFrame; .shape[0] é o número de linhas e .shape[1] é o número de colunas.[1]
print(f"Linhas: {dados_cogumelos.shape[0]}\nColunas: {dados_cogumelos.shape[1]}")

## Visualização dos dados presente no dataset

In [ ]:
# Visualização da contagem de cogumelos venenosos e não venenosos em relação à raiz do caule.
# O countplot exibe o número de ocorrências para cada categoria dentro de uma variável.
sns.countplot(data=dados_cogumelos, x="stalk-root", hue="poisonous")

In [ ]:
# Visualização da contagem de cogumelos por formato de chapéu e se são venenosos ou não.
fig, ax =plt.subplots(1,3, figsize=(15,5))  # Cria uma figura com 3 subplots (gráficos) lado a lado.
sns.countplot(x="cap-shape", hue='poisonous', data=dados_cogumelos, ax=ax[0])  # Gráfico de contagem para analisar a distribuição de cogumelos por formato de chapéu.
sns.countplot(x="cap-surface", hue='poisonous', data=dados_cogumelos, ax=ax[1])
sns.countplot(x="cap-color", hue='poisonous', data=dados_cogumelos, ax=ax[2])
fig.tight_layout()
fig.show()

In [ ]:
# Visualização da contagem de cogumelos por presença de hematomas e se são venenosos ou não.
fig, ax =plt.subplots(1,2, figsize=(15,5))
sns.countplot(x="bruises", hue='poisonous', data=dados_cogumelos, ax=ax[0])
sns.countplot(x="odor", hue='poisonous', data=dados_cogumelos, ax=ax[1])
fig.tight_layout()
fig.show()

In [ ]:
# Visualização da contagem de cogumelos por apego de lamela e se são venenosos ou não.
fig, ax =plt.subplots(1,4, figsize=(20,5))
sns.countplot(x="gill-attachment", hue='poisonous', data=dados_cogumelos, ax=ax[0])
sns.countplot(x="gill-spacing", hue='poisonous', data=dados_cogumelos, ax=ax[1])
sns.countplot(x="gill-size", hue='poisonous', data=dados_cogumelos, ax=ax[2])
sns.countplot(x="gill-color", hue='poisonous', data=dados_cogumelos, ax=ax[3])
fig.tight_layout()
fig.show()

## Transformação dos dados

In [ ]:
# Função para codificar as variáveis categóricas em valores numéricos usando LabelEncoder.
from sklearn.preprocessing import LabelEncoder

def encode_labels(df, columns):  # Define uma função para codificar as colunas categóricas usando LabelEncoder.
    """
    Aplica LabelEncoder em múltiplas colunas de um DataFrame.

    Parâmetros:
    - df: DataFrame a ser processado.
    - columns: Lista de colunas a serem codificadas.

    Retorno:
    - DataFrame com as colunas codificadas.
    """
    label_encoders = {}  # Cria um dicionário para armazenar os codificadores para cada coluna.

    for column in columns:
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

    return df, label_encoders


### Codificação de Variáveis Categóricas

In [ ]:
# Aplicação da codificação em todas as colunas do dataset.
# Seleciona todas as colunas do dataset para codificação.
# Aplica a função de codificação a todas as colunas categóricas.

# columns_to_encode = dados_cogumelos.columns
# df_encoded, label_encoders = encode_labels(dados_cogumelos, columns_to_encode)

In [ ]:
# Transformação das colunas categóricas em variáveis dummy (one-hot encoding).
# Converte as variáveis categóricas em várias colunas binárias.

# df_encoded = pd.get_dummies(dados_cogumelos, dtype='int')
# df_encoded.head()

### Divisão dos Dados em Treino e Teste

In [ ]:
# Divisão dos dados em features (X) e target (y).

X = dados_cogumelos.drop(['poisonous'], axis=1)

columns_to_encode = X.columns # pega as colunas
X, label_encoders = encode_labels(X, columns_to_encode) # Aplicação da codificação em todas as colunas do dataset, exceto a target.
X

In [ ]:
# Definição do target (variável dependente) como a coluna 'poisonous'.
y = dados_cogumelos['poisonous']
y

In [ ]:
# Divisão dos dados em conjuntos de treino e teste.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

# Treinamento do Modelo de Árvore de Decisão

Com os dados preparados, agora treinaremos o modelo de Árvore de Decisão. Usaremos os dados de treino para ajustar o modelo e, em seguida, faremos previsões com os dados de teste para avaliar o desempenho do modelo.


In [ ]:
# Treinamento do modelo de Árvore de Decisão e previsão com os dados de teste.
from sklearn import tree
from sklearn.metrics import accuracy_score

dtc = tree.DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=0, min_samples_split=3)
dtc.fit(X_train, y_train)
pred = dtc.predict(X_test)

In [ ]:
# Cálculo da acurácia do modelo.

accuracy_score(y_true=y_test, y_pred=pred)

In [ ]:
# Identificação das importâncias das features no modelo de Árvore de Decisão.
dtc.feature_importances_

## Visualização da Árvore de Decisão

Uma das vantagens da Árvore de Decisão é que podemos visualizar como o modelo está tomando suas decisões. Vamos plotar a árvore para entender melhor o processo de decisão e identificar quais características estão influenciando mais as decisões.


In [ ]:
# Visualização gráfica da Árvore de Decisão gerada.
import graphviz
dot_data = tree.export_graphviz(dtc, feature_names=X.columns.values, class_names=['Edible', 'Poisonous'], filled=True )
graphviz.Source(dot_data)

# Avaliação do Modelo

Após treinar o modelo, é essencial avaliar sua performance. Vamos utilizar várias métricas para isso: acurácia, precisão, recall, F1-Score e especificidade. Essas métricas nos darão uma visão completa de como o modelo está performando.


### Matriz de Confusão

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(dtc, X_test, y_test)
plt.grid(False)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Relatório de classificação
print('\nRelatório de Classificação:\n', classification_report(y_test, pred))

#### Cálculo de Precisão, Recall e F1-Score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Calculando a precisão (Precision) de cada classe
precisao_por_classe = precision_score(y_test, pred, average=None)

# Calculando o recall (Sensibilidade) de cada classe
recall_por_classe = recall_score(y_test, pred, average=None)

# Calculando o F1-Score de cada classe
f1_por_classe = f1_score(y_test, pred, average=None)

# Exibindo as métricas para cada classe
for i in range(len(precisao_por_classe)):
    print(f'Classe {i}:')
    print(f' Precisão: {precisao_por_classe[i]:.2f}')
    print(f' Recall: {recall_por_classe[i]:.2f}')
    print(f' F1-Score: {f1_por_classe[i]:.2f}')
    print()


### 4.6 Validação Cruzada

A validação cruzada permite avaliar a performance do modelo de forma mais robusta. Ao medir a acurácia tanto nos conjuntos de treino quanto nos de teste, podemos observar se o modelo está superajustando aos dados de treino.

Vamos aplicar a validação cruzada com 5 folds e extrair as acurácias para ambos os conjuntos.

In [ ]:
from sklearn.model_selection import cross_validate
# Aplicando a validação cruzada com 10 folds
resultados = cross_validate(dtc, X, y, cv=10, scoring='accuracy', return_train_score=True)

# Extraindo as métricas
acuracia_treino = resultados['train_score']
acuracia_teste = resultados['test_score']

# Exibindo os resultados de forma elegante
print("="*50)
print("Resultados da Validação Cruzada com 10 Folds")
print("="*50)
for i in range(len(acuracia_treino)):
    print(f"Fold {i+1}:")
    print(f"    Acurácia no Treino: {acuracia_treino[i]:.2f}")
    print(f"    Acurácia no Teste: {acuracia_teste[i]:.2f}")
    print("-"*50)

print(f"Acurácia Média no Treino: {acuracia_treino.mean():.2f} ± {acuracia_treino.std():.2f}")
print(f"Acurácia Média no Teste: {acuracia_teste.mean():.2f} ± {acuracia_teste.std():.2f}")
print("="*50)


# Interpretação dos Resultados

Agora que temos todas as métricas calculadas, vamos interpretá-las. Entenderemos o que cada métrica representa e como o modelo está performando em termos de precisão, recall e F1-Score. Isso nos ajudará a avaliar se o modelo é adequado para o problema que estamos tentando resolver.


### Utilizando apenas os atributos com maior ganha de informações
#### Reaplicando a árvore de decisão

In [ ]:
dados_atributos_import = dados_cogumelos[['gill-color', 'population', 'spore-print-color']]

columns_to_encode = dados_atributos_import.columns
X_atributos_import, label_encoders = encode_labels(dados_atributos_import, columns_to_encode)
X_atributos_import

In [ ]:
# Divisão dos dados de atributos importantes em conjuntos de treino e teste.
X_train, X_test, y_train, y_test = train_test_split(X_atributos_import, y, test_size=0.20, random_state=0)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Treinamento de um novo modelo de Árvore de Decisão usando os atributos mais importantes.
tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=0, min_samples_split=3)
tree.fit(X_train, y_train)
pred = tree.predict(X_test)

In [ ]:
# Cálculo da acurácia do novo modelo com atributos selecionados.
accuracy_score(y_true=y_test, y_pred=pred)

In [ ]:
# Predição de novos dados usando o modelo treinado.
previsores = tree.predict([[5,4,0], [2,0,7],[4,2,6], [5,2,0]])
previsores

# Conclusão

Neste notebook, exploramos o algoritmo de Árvore de Decisão, desde a preparação dos dados até a avaliação do modelo. Discutimos as principais métricas e visualizamos como as decisões são tomadas. Como próximos passos, podemos explorar outros algoritmos ou realizar ajustes finos no modelo atual para melhorar sua performance.
